In [ ]:
# trec06c 是一个公开的垃圾邮件语料库，由国际文本检索会议提供，分为英文数据集（trec06p）和中文数据集（trec06c），
# 其中所含的邮件均来源于真实邮件保留了邮件的原有格式和内容。
# 整个数据集一共有 64620 个样本，第一列代表的是是否是垃圾邮件，标记 spam 是垃圾邮件，标记 ham 是正常邮件。
# 第二列则是邮件内容的文本文件的路径

In [ ]:
# 下载数据集
!wget -nc "https://labfile.oss.aliyuncs.com/courses/1176/trec06c.zip"  # 下载数据
!unzip -o trec06c.zip

In [1]:
# 加载数据集
import pandas as pd
import numpy as np
data = pd.read_table('trec06c/full/index', header=None, encoding='gb2312', delim_whitespace=True)
data.head()

/tmp/ipykernel_1018930/1087613909.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_table is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_table('trec06c/full/index', header=None, encoding='gb2312', delim_whitespace=True)


,0,1
0,spam,../data/000/000
1,ham,../data/000/001
2,spam,../data/000/002
3,spam,../data/000/003
4,spam,../data/000/004


In [2]:
# 0列标签进行数值化
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data[0] = le.fit_transform(data[0])
# 打印labels列的字符串与数值对应关系
print("编码前后的对应关系：")
for i, class_name in enumerate(le.classes_):
    print(f"原始标签: '{class_name}' -> 编码值: {i}")

编码前后的对应关系：
原始标签: 'ham' -> 编码值: 0
原始标签: 'spam' -> 编码值: 1


In [3]:
df = data.replace(regex=['\..'], value='trec06c')  # 去掉路径中的点号
df = df.sample(len(df), random_state=42)[:10000]  # 为了加快训练速度，这里只取10000条数据
df.groupby(0).size()  # 查看各类别样本数量

0
0    3344
1    6656
dtype: int64

In [ ]:
# 查看文件路径对应的邮件内容
# 邮件由两部分组成，第一部分包含了邮件的信息，例如发件人，标题等等，第二部分才是邮件正文。
# 这些文件不是 UTF-8 编码的，所以需要将其转为 UTF-8 编码
!cat "trec06c/data/000/002" | iconv -f gbk -t utf-8  # 使用iconv进行编码转换

In [ ]:
# 需要数据预处理
# 转换源数据编码格式为 UTF-8 格式。
# 过滤字符：去除所有非中文字符，如标点符号、英文字符、数字、网站链接等特殊字符。
# 过滤停用词。
# 对邮件内容进行分词处理。

In [ ]:
# 在中文中，有很多的非实意词语或者其他并没有实际作用的词语，这些词语必须在分词之后进行过滤，这个环节也就是过滤停用词
# 通过一个预先设定的停用词表，然后判断是否是停用词来过滤分词结果。另外如果是空行，则需要剔除。分词之后需剔除只有一个字的结果

In [ ]:
# 下载停用词表
!wget -nc "https://labfile.oss.aliyuncs.com/courses/1176/stopwords.txt"

In [4]:
# 编写分词和过滤停用词的函数
import jieba
import re

def tokenize_and_remove_stopwords(text, stopwords):
    # 分词
    words = jieba.lcut(text)
    # 过滤停用词和单字词
    filtered_words = [word for word in words if word not in stopwords and len(word) > 1 and re.match(r'^[\u4e00-\u9fa5]+$', word)]
    return filtered_words
# 读取停用词表
with open('stopwords.txt', 'r', encoding='utf-8') as f:
    stopwords = set(f.read().splitlines())
# 测试分词和过滤停用词函数
sample_text = "这是一个用于测试分词和停用词过滤的示例文本。"
print(tokenize_and_remove_stopwords(sample_text, stopwords))

/root/miniconda3/envs/universe/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.969 seconds.
Prefix dict has been built successfully.


['这是', '用于', '测试', '分词', '用词', '过滤', '示例', '文本']


In [5]:
# 对邮件内容进行格式转换及分词处理
def load_and_process_email(file_path, stopwords):
    with open(file_path, 'r', encoding='gbk', errors='ignore') as f:
        content = f.read()
    # 分离邮件头和邮件正文
    parts = content.split('\n\n', 1)
    body = parts[1] if len(parts) > 1 else ''
    # 分词并过滤停用词
    tokens = tokenize_and_remove_stopwords(body, stopwords)
    return tokens
# 测试加载和处理邮件函数
sample_file_path = "trec06c/data/000/002"
print(load_and_process_email(sample_file_path, stopwords))

['尊敬', '公司', '财务', '经理', '负责人', '您好', '深圳', '金海', '实业', '有限公司', '广州', '东莞', '省市', '分公司', '我司', '社会关系', '实力', '每月', '进项', '多出', '项少', '现有', '一部分', '发票', '优惠', '对外', '代开', '税率', '增值税', '发票', '国税', '地税', '运输', '广告', '普通发票', '税点', '数目', '大小', '来衡', '优惠', '希望', '公司', '商家', '来电', '商谈', '合作', '公司', '郑重', '承诺', '所用', '票据', '税务局', '验证', '抵扣', '来电', '进一步', '商谈', '电话', '小时', '服务', '信箱', '联系人', '海南', '顺祝', '商祺', '深圳市', '金海', '实业', '有限公司']


In [6]:
# 对全部样本进行分词操作
from tqdm import tqdm

df['tokens'] = df[1].apply(lambda x: load_and_process_email(x, stopwords))
df.head()

,0,1,tokens
25228,0,trec06c/data/084/028,"[几天, 闹矛盾, 胡闹, 不道, 原谅, 真的, 过头, 不放过, 三件, 事才, 原谅,..."
10085,1,trec06c/data/033/185,"[整改, 对策, 课程, 收益, 系统, 学习, 测试方法, 对策, 理论, 实践, 知识,..."
45208,0,trec06c/data/150/208,"[指名道姓, 前前, 一段, 交大, 陈贺, 出生, 身高, 湖南大学, 有机化学, 专业,..."
16700,1,trec06c/data/055/200,"[尊敬, 商家, 朋友, 您好, 深圳市, 华龙, 公司, 多家, 公司, 委托, 向外, ..."
64536,0,trec06c/data/215/036,"[昨儿, 清华, 考研, 情况, 谈谈, 见闻, 昨天, 毛毛雨, 润物细无声, 半截, 裤..."


In [7]:
# 使用 gensim 提供的 Word2vec API把分词结果编码为可以输入算法的向量
from gensim.models import Word2Vec
from tqdm import tqdm_notebook

import warnings
warnings.filterwarnings("ignore")

# 导入保存的分词数组
data = df['tokens']

# 训练浅层的Word2Vec模型
w2v_model = Word2Vec(vector_size=100, min_count=10)
w2v_model.build_vocab(data)
w2v_model.train(data, total_examples=w2v_model.corpus_count, epochs=5)
w2v_model

In [9]:

def sum_vec(text):
    # 对每个句子进行词向量求和计算
    vec = np.zeros(100).reshape((1, 100))
    for word in text:
        try:
            # 使用 wv 属性来获取词向量
            vec += w2v_model.wv[word].reshape((1, 100))  # 修改这里：将 w2v_model[word] 改为 w2v_model.wv[word]
        except KeyError:
            continue  # 如果单词不在词汇表中，跳过
    return vec

# 将词向量保存为 Ndarray
data_vec = np.concatenate([sum_vec(z) for z in tqdm_notebook(data)])
data_vec.shape

  0%|          | 0/10000 [00:00<?, ?it/s]

(10000, 100)

In [10]:
from sklearn.model_selection import  cross_val_predict, KFold
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

features = data_vec
labels = df[0].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_predict(GaussianNB(), features, labels, cv=kf)
print("最终模型评估：")
print("准确率:", accuracy_score(labels, scores))


最终模型评估：
准确率: 0.5614


In [12]:
from sklearn.naive_bayes import BernoulliNB
model = BernoulliNB()
scores = cross_val_predict(BernoulliNB(), features, labels, cv=kf)
print("最终模型评估：")
print("准确率:", accuracy_score(labels, scores))

最终模型评估：
准确率: 0.9459
